# 1c″ - hư hại ngữ pháp có dự đoán F1 không, ở n=1003

Ở n=200 reader 7B, LLMLingua-2 cho ρ = **−0.129**, CI [−0.26, **+0.005**] - đúng
dấu giả thuyết dự đoán nhưng thiếu đúng một chút để đạt ý nghĩa. Đây là tín hiệu
cuối còn đáng theo cho lập luận nhân quả ở mục 2.3.

**Kernel này KHÔNG cần GPU.** `vimqa_full_7b.json` đã có F1 từng câu cho cả 1.003
câu; script chỉ đọc `per_row` rồi tính lại tỉ số hư từ bằng LLMLingua-2
(BERT-base) và bge-m3. Cỡ mẫu **gấp 5 lần** n=200.

File kết quả lấy từ Dataset `nhantrnh/itercomp-vimqa-full-7b` gắn sẵn - kernel
**không** tự chạy lại bảng chính, vì việc đó cần GPU và một kernel CPU sẽ bị huỷ
giữa chừng.

In [ ]:
!pip install -q llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -2


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats', 'budget']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'✓ tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean, budget_filter
print('cwd:', os.getcwd(), '| scripts/:', os.path.isdir('scripts'))
print(f'✓ đủ {len(NEED)} module')

In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
import subprocess, sys, os, time

def run_stream(cmd, logfile, env=None):
    """In output NGAY thay vì giữ tới lúc xong.

    LLMLingua-2 trên 1.003 ngữ cảnh chạy hàng chục phút; capture_output nghĩa là
    ngồi nhìn màn hình trống, không biết treo hay đang chạy. Cũng gộp stderr vào
    stdout, vì một lần trước stderr bị nuốt và script chết lặng lẽ.
    """
    t0 = time.time()
    with open(logfile, 'w') as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env=env)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

## 1. Lấy file kết quả n=1003

In [ ]:
import os, sys, glob, shutil
os.makedirs('results', exist_ok=True)
OUT = 'results/vimqa_full_7b.json'
if not os.path.exists(OUT):
    hits = glob.glob('/kaggle/input/**/vimqa_full_7b.json', recursive=True)
    if not hits:
        raise SystemExit(
            'Thiếu vimqa_full_7b.json.\n'
            'Add Input > Datasets > nhantrnh/itercomp-vimqa-full-7b')
    shutil.copy(hits[0], OUT)
    print(f'lấy từ input: {hits[0]}')
print(f'✓ {OUT}')

## 2. Chạy 1c″

In [ ]:
env = dict(os.environ, ITERCOMP_SCORER_DEVICE='cpu')
rc = run_stream([sys.executable, '-u', 'scripts/damage_vs_f1.py',
                 '--eval-json', 'results/vimqa_full_7b.json',
                 '--out', 'results/damage_vs_f1_vimqa_full.json'],
                'results/log_damage_full.txt', env=env)
assert rc == 0, f'thất bại (rc={rc})' 

## 3. So hai cỡ mẫu

In [ ]:
import json
new = json.load(open('results/damage_vs_f1_vimqa_full.json'))
OLD = {'itercomp':   {'rho': -0.022, 'lo': -0.16, 'hi': +0.11},
       'llmlingua2': {'rho': -0.129, 'lo': -0.26, 'hi': +0.01}}

def fmt(r, lo, hi):
    return f'rho {r:+.3f} [{lo:+.2f}, {hi:+.2f}]'

print(f"{'phương pháp':12s} {'n=200':>26s} {'n=1003':>26s}")
for m, o in OLD.items():
    d = new['summary'][m]
    print(f"{m:12s} {fmt(o['rho'], o['lo'], o['hi']):>26s}"
          f" {fmt(d['rho'], d['ci_lo'], d['ci_hi']):>26s}")

sig = [m for m in OLD
       if new['summary'][m]['ci_hi'] < 0 or new['summary'][m]['ci_lo'] > 0]
print()
if sig:
    print(f"CÓ tương quan ở n=1003 ({', '.join(sig)}).")
    print("Cỡ mẫu đúng là thứ thiếu - mục 2.3 viết được lập luận nhân quả.")
else:
    print("VẪN không tương quan ở n=1003, gấp 5 lần cỡ mẫu cũ.")
    print("Kết quả âm được CHỐT: hư hại ngữ pháp không giải thích được F1 sụt.")

## 4. Tải về

In [ ]:
import shutil, os
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z = shutil.make_archive(os.path.join(BASE, 'results_1cdouble'), 'zip', 'results')
print(f'✓ {z}  ({os.path.getsize(z)/1e6:.1f} MB)')